# Laboratorio 4: regresión logística binaria con PyTorch

**Autor:** Nataniel Mauricio Arapa Estrada  
**Materia:** SIS420 — Inteligencia Artificial I

El objetivo es predecir `seasonal_vaccine`. La regresión logística se representa con una capa `nn.Linear` y se entrena con `BCEWithLogitsLoss`, que combina de forma estable la sigmoide y la entropía cruzada binaria.

Pandas se limita a cargar, imputar y codificar el CSV. La división estratificada, normalización, modelo, optimización, inferencia y métricas se implementan con PyTorch; no se utiliza scikit-learn.


## 1. Configuración y carga


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def buscar_archivo(nombre, carpeta_lab):
    bases = [
        Path.cwd(), Path.cwd() / carpeta_lab, Path.cwd().parent / carpeta_lab,
        Path("/content/drive/MyDrive/Colab Notebooks/machine learning/datasets"),
        Path("/content/gdrive/MyDrive/Colab Notebooks/machine learning/datasets"),
    ]
    for base in bases:
        candidato = base / nombre
        if candidato.exists():
            return candidato
    raise FileNotFoundError(f"No se encontró {nombre}. Colócalo junto al notebook o en Drive.")


ruta_h1n1 = buscar_archivo("H1N1_Flu_Vaccines.csv", "Lab4")
datos_crudos = pd.read_csv(ruta_h1n1)
print(f"PyTorch {torch.__version__} | Dispositivo: {DEVICE}")
print(f"Dataset: {ruta_h1n1}")
print(f"Dimensiones originales: {datos_crudos.shape}")


## 2. Limpieza y codificación

- Se elimina `respondent_id`, que es solo un identificador.
- Se elimina `h1n1_vaccine`: conocer otra decisión de vacunación podría introducir una señal demasiado directa para el objetivo de este ejercicio.
- Se conserva el umbral original de 12 000 valores ausentes.
- Las columnas numéricas se imputan con la mediana y las categóricas con la moda. La interpolación por orden de fila del cuadernillo original no era apropiada, porque las filas son personas independientes.


In [ ]:
objetivo = "seasonal_vaccine"
limite_nulos = 12_000
muchas_ausencias = [
    col for col, cantidad in datos_crudos.isna().sum().items()
    if cantidad > limite_nulos and col != objetivo
]

eliminar = list(dict.fromkeys(["respondent_id", "h1n1_vaccine", *muchas_ausencias]))
X_df = datos_crudos.drop(columns=eliminar, errors="ignore").drop(columns=[objetivo]).copy()
y = torch.tensor(datos_crudos[objetivo].to_numpy(), dtype=DTYPE)

columnas_numericas = X_df.select_dtypes(include="number").columns
columnas_categoricas = X_df.select_dtypes(exclude="number").columns
X_df[columnas_numericas] = X_df[columnas_numericas].fillna(
    X_df[columnas_numericas].median()
)
for columna in columnas_categoricas:
    moda = X_df[columna].mode(dropna=True)
    X_df[columna] = X_df[columna].fillna(
        moda.iloc[0] if not moda.empty else "desconocido"
    )

X_df = pd.get_dummies(X_df, drop_first=True, dtype="float32")
nombres = X_df.columns.tolist()
X = torch.tensor(X_df.to_numpy(dtype="float32"), dtype=DTYPE)

conteos = torch.bincount(y.long(), minlength=2)
print(f"Columnas eliminadas por ausencias: {muchas_ausencias}")
print(f"Nulos restantes: {int(X_df.isna().sum().sum())}")
print(f"Características finales: {X.shape[1]}")
print(f"Clase 0: {conteos[0].item():,} | Clase 1: {conteos[1].item():,}")
print(f"Porcentaje positivo: {100 * y.mean().item():.2f}%")


## 3. División estratificada y normalización


In [ ]:
def division_estratificada(X, y, proporcion_prueba=0.20, semilla=SEED):
    generador = torch.Generator().manual_seed(semilla)
    train_partes, test_partes = [], []
    for clase in torch.unique(y):
        indices_clase = torch.where(y == clase)[0]
        indices_clase = indices_clase[
            torch.randperm(len(indices_clase), generator=generador)
        ]
        n_test = int(len(indices_clase) * proporcion_prueba)
        test_partes.append(indices_clase[:n_test])
        train_partes.append(indices_clase[n_test:])

    idx_train = torch.cat(train_partes)
    idx_test = torch.cat(test_partes)
    idx_train = idx_train[torch.randperm(len(idx_train), generator=generador)]
    idx_test = idx_test[torch.randperm(len(idx_test), generator=generador)]
    return X[idx_train], X[idx_test], y[idx_train], y[idx_test]


X_train, X_test, y_train, y_test = division_estratificada(X, y)
media = X_train.mean(dim=0, keepdim=True)
desviacion = X_train.std(dim=0, unbiased=False, keepdim=True).clamp_min(1e-8)
X_train_n = (X_train - media) / desviacion
X_test_n = (X_test - media) / desviacion

print(f"Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}")
print(f"Positivos train: {100 * y_train.mean().item():.2f}%")
print(f"Positivos test:  {100 * y_test.mean().item():.2f}%")


## 4. Modelo y entrenamiento

El modelo produce *logits* $z=Xw+b$. `BCEWithLogitsLoss` evalúa directamente esos logits; la función sigmoide se aplica únicamente al inferir probabilidades.


In [ ]:
torch.manual_seed(SEED)
modelo = nn.Linear(X_train_n.shape[1], 1).to(DEVICE)
criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.AdamW(modelo.parameters(), lr=0.01, weight_decay=1e-4)
generador = torch.Generator().manual_seed(SEED)
cargador = DataLoader(
    TensorDataset(X_train_n, y_train),
    batch_size=512,
    shuffle=True,
    generator=generador,
)

historial = []
epocas = 80
for epoca in range(epocas):
    modelo.train()
    perdida_acumulada = 0.0
    for xb, yb in cargador:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizador.zero_grad()
        logits = modelo(xb).squeeze(1)
        perdida = criterio(logits, yb)
        perdida.backward()
        optimizador.step()
        perdida_acumulada += perdida.item() * len(yb)
    historial.append(perdida_acumulada / len(X_train_n))

    if epoca == 0 or (epoca + 1) % 20 == 0:
        print(f"Época {epoca + 1:>3}/{epocas} | BCE: {historial[-1]:.5f}")


## 5. Evaluación sin scikit-learn


In [ ]:
modelo.eval()
with torch.inference_mode():
    logits_test = modelo(X_test_n.to(DEVICE)).cpu().squeeze(1)
    probabilidades = torch.sigmoid(logits_test)
    predicciones = (probabilidades >= 0.5).long()

reales = y_test.long()
matriz = torch.bincount(reales * 2 + predicciones, minlength=4).reshape(2, 2)
tn, fp = matriz[0].tolist()
fn, tp = matriz[1].tolist()

accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
especificidad = tn / max(tn + fp, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)

print("Matriz de confusión [[TN, FP], [FN, TP]]")
print(matriz)
print(f"Accuracy:      {accuracy:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"Especificidad: {especificidad:.4f}")
print(f"F1:            {f1:.4f}")

ejemplos = pd.DataFrame({
    "probabilidad": probabilidades[:10].tolist(),
    "predicción": predicciones[:10].tolist(),
    "real": reales[:10].tolist(),
})
display(ejemplos.round(4))


## 6. Interpretación de coeficientes y gráficas


In [ ]:
pesos = modelo.weight.detach().cpu().squeeze(0)
k = min(8, len(nombres))
indices_positivos = torch.topk(pesos, k=k).indices
indices_negativos = torch.topk(-pesos, k=k).indices

influencias = pd.DataFrame({
    "Aumentan la probabilidad": [nombres[i] for i in indices_positivos.tolist()],
    "peso +": pesos[indices_positivos].tolist(),
    "Reducen la probabilidad": [nombres[i] for i in indices_negativos.tolist()],
    "peso -": pesos[indices_negativos].tolist(),
})
display(influencias.round(4))

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
ejes[0].plot(historial)
ejes[0].set(title="Entrenamiento", xlabel="Época", ylabel="BCE")
ejes[0].grid(alpha=0.3)

ejes[1].hist(
    probabilidades[reales == 0].tolist(), bins=25, alpha=0.65, label="Clase real 0"
)
ejes[1].hist(
    probabilidades[reales == 1].tolist(), bins=25, alpha=0.65, label="Clase real 1"
)
ejes[1].axvline(0.5, color="black", linestyle="--", label="Umbral")
ejes[1].set(
    title="Probabilidades en prueba", xlabel="P(vacuna estacional = 1)", ylabel="Casos"
)
ejes[1].legend()
plt.tight_layout()
plt.show()


## Conclusiones

- La variable positiva representa aproximadamente 46.6% de los casos: el conjunto es razonablemente balanceado, aunque no exactamente 50/50.
- `BCEWithLogitsLoss` evita los problemas numéricos de calcular manualmente `log(sigmoid(z))`.
- Accuracy se acompaña de precision, recall, especificidad y F1; una sola métrica no describe todos los tipos de error.
- Los pesos indican asociación dentro de este modelo lineal, no causalidad.
